# Local model vs. hosted portal API

Runs the same bundled sample image through **this repo's local DINOv3 point
model** and through **the hosted InsightML portal API**, side by side.

> The hosted `count_wheat` endpoint currently serves the *previous-generation*
> VGG density model, not the DINOv3 point model this repo trains — count-only,
> no per-plant coordinates. See `docs/portal.md` for the full picture; this
> notebook exists to make that gap visible on one concrete image, not to
> validate one model against the other.

Needs, to run both halves:
- a local checkpoint at `../weights/decoder_best.pt` (train one with
  `notebooks/training.ipynb`, or drop one in) — the local-model cells degrade
  to a skip message if it's missing;
- a portal API key (`.env` with `INSIGHTML_API_KEY`, see `.env.example`) — the
  portal cells degrade to a skip message if it's missing.

Either half can be skipped; the rest of the notebook still runs.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import torch
from PIL import Image

from cropcounter import decode_in_bounds, load_checkpoint, predict_prob, resolve_device
from cropcounter.portal import AuthError, MarkerDetectionError, PortalClient, RateLimitError

device = resolve_device()
print(torch.__version__, device)

## Sample image

In [ ]:
SAMPLE_IMAGE = sorted(Path("../examples/data/val/images").glob("*.jpg"))[0]
print(SAMPLE_IMAGE.resolve())
Image.open(SAMPLE_IMAGE)

## Local model

This repo's DINOv3 pyramid-decoder point model, run from a local checkpoint.

In [ ]:
CKPT_PATH = Path("../weights/decoder_best.pt")
TAU = 0.35

local_count = None
if CKPT_PATH.exists():
    from cropcounter import CropTileDataset, records_from_folder

    model, cfg = load_checkpoint(CKPT_PATH, device, weights_dir=Path("../weights"))
    model.eval()

    recs = [r for r in records_from_folder(SAMPLE_IMAGE.parent) if r.name == SAMPLE_IMAGE.name]
    ds = CropTileDataset(recs, SAMPLE_IMAGE.parent, train=False,
                         output_stride=cfg.output_stride, sigma=cfg.sigma)
    item = ds[0]
    prob = predict_prob(model, item["image"], device)
    local_points, _ = decode_in_bounds(prob, recs[0].width, recs[0].height, tau=TAU,
                                       k=cfg.k, nms_radius=cfg.nms_radius,
                                       output_stride=cfg.output_stride)
    local_count = len(local_points)
    print(f"local model predicted count: {local_count}")
else:
    print(f"no checkpoint at {CKPT_PATH.resolve()} yet — train one with "
          f"notebooks/training.ipynb, or drop decoder_best.pt into weights/, "
          f"then re-run this cell")

## Hosted portal API

See `docs/portal.md` — currently the previous-generation VGG density model, count-only.

**On the bundled sample this call is expected to fail.** `examples/data` holds
pre-cropped quadrat interiors, and the hosted endpoint locates the quadrat via the
4 printed ArUco corner markers that cropping removed — so it answers HTTP 422 /
`error_code` 202 and the cell below prints the `MarkerDetectionError` instead of a
count. Point `SAMPLE_IMAGE` at your own uncropped quadrat photo (all 4 markers
visible) to get a real side-by-side comparison.

In [ ]:
portal_count = None
portal_metadata = None
try:
    client = PortalClient()  # key from $INSIGHTML_API_KEY, env=prod
except AuthError as exc:
    client = None
    print(f"skipping portal call: {exc}")

if client is not None:
    try:
        result = client.count_wheat(SAMPLE_IMAGE)
        portal_count = result.predicted_count
        portal_metadata = result.metadata
        time_str = f"{result.inference_time_seconds:.2f}s" if result.inference_time_seconds else "n/a"
        print(f"portal predicted count: {portal_count}  (inference {time_str})")
    except MarkerDetectionError as exc:
        print(f"portal could not detect the quadrat markers: {exc}")
    except RateLimitError as exc:
        print(f"portal rate-limited: {exc}")

## Side by side

In [ ]:
print(f"{'local (this repo, DINOv3 point model)':45s} {local_count if local_count is not None else '—'}")
print(f"{'hosted portal (previous-gen VGG density model)':45s} {portal_count if portal_count is not None else '—'}")
if portal_metadata:
    print(f"\nportal metadata: {portal_metadata}")